### Ejercicio, resumen webs

Ollama debe estar corriendo en local:  
[http://localhost:11434/](http://localhost:11434/)



In [1]:
# importaciones

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [2]:
# Constantes

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [20]:
# Crea un mensaje

messages = [
    {"role": "user", "content": "Describe algunas de las aplicaciones que puede generar la IA"}
]

In [21]:
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

In [22]:
# Comprobamos que el modelo esta cargadfo

!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [23]:
response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()['message']['content'])

La inteligencia artificial (IA) tiene un amplio rango de aplicaciones en diversas áreas, desde la vida diaria hasta la industria y la sociedad en general. A continuación, se presentan algunas de las aplicaciones más destacadas de la IA:

**Aplicaciones en la vida diaria**

1. **Asistentes virtuales**: como Siri, Alexa o Google Assistant, que pueden realizar tareas como programar recordatorios, responder a preguntas y controlar dispositivos inteligentes.
2. **Recomendaciones personalizadas**: como las recomendaciones de música, películas o productos en línea, que se basan en tus preferencias y comportamientos.
3. **Navegación inteligente**: como Google Maps, que utiliza la IA para proporcionar rutas más eficientes y recomendaciones de lugares interesantes.

**Aplicaciones en la salud**

1. **Análisis médicos**: la IA puede ayudar a diagnosticar enfermedades y prescribir tratamientos personalizados.
2. **Monitoreo de la salud**: la IA puede monitorear la salud de los pacientes y detectar

#### Ahora con el paqquete de ollama 

In [24]:
import ollama

response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

La inteligencia artificial (IA) es una tecnología en constante evolución, y sus aplicaciones son amplias y variadas. A continuación, te presento algunas de las aplicaciones más comunes y significativas de la IA:

**Aplicaciones en el ámbito empresarial**

1. **Análisis predictivo**: La IA puede analizar grandes cantidades de datos para predecir tendencias y patrones en el mercado, ayudando a los negocios a tomar decisiones informadas.
2. **Automatización del trabajo**: La IA puede automatizar tareas repetitivas y tediosas, liberando tiempo y recursos para que los empleados se enfoquen en aspectos más creativos y estratégicos.
3. **Marketing personalizado**: La IA puede analizar datos de comportamiento y preferencias para ofrecer experiencias de marketing personalizadas y relevantes a los clientes.

**Aplicaciones en la salud**

1. **Detección de enfermedades**: La IA puede ayudar a detectar enfermedades tempranas mediante el análisis de imágenes médicas, ruido de cerebro o otros datos 

# Scrapear y resumir Webs

In [8]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [9]:
MODEL = "llama3.2"

In [25]:
class Website:
  
    url: str
    title: str
    text: str

    def __init__(self, url):
  
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No se ha encontrado título"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [11]:
ed = Website("https://www.xataka.com")
print(ed.title)
print(ed.text)

Xataka - Tecnología y gadgets, móviles, informática, electrónica
Webedia
Xataka
Menú
Nuevo
ANÁLISIS
XATAKA SELECCIÓN
MÓVILES
CIENCIA
GUÍAS DE COMPRA
MOVILIDAD
MAGNET
WhatsApp
Tiktok
Telegram
Instagram
Youtube
Twitter
PARTNERS
Life´s Good by LG
muévete con Renault
Innovación Volvo
Make it realme
HP impulsa tu pyme
La privacidad de WhatsApp parecía a prueba de bombas. Hasta que un fiscal del estado intentó borrar mensajes incriminatorios
Euro Digital: qué es, cómo va a funcionar y cuándo va a llegar
Estados Unidos impide a Huawei fabricar más PCs con Windows. Su respuesta: lanzar su primer portátil con Linux
HOY SE HABLA DE
China
Nintendo Switch 2
Fallout
Tesla
Arabia Saudí
iPhone 16
Android Auto
IA
OpenAI
Mapa
Lo último
Hace tres meses me compré el Mac mini M4. Estoy contento con él, pero ojalá hubiera esperado a su nueva oferta
Xataka Selección
Un precio muy ajustado para uno de los dispositivos de Apple con mejor relación calidad-precio.
Leer más »
1
comentarios
Alberto García
2025-05

### Tipos de prompts

In [27]:
system_prompt = "Eres un asistente que analiza contenidos web  \
y hace resumenes sobre estas webs ignorando el texto que este relacionado con navegacion. \
Respond in markdown."

In [28]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "El contenido de esta pagina web es; \
hazme un pequeño resumen en formato markdown. \
SI hay noticias o publicidad, resumela tambien.\n\n"
    user_prompt += website.text
    return user_prompt

##### Tanto Ollama como otros modelos esperan este formato:

In [14]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [15]:
def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [16]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [17]:
display_summary("https://www.xataka.com")

Aquí tienes algunos artículos de los últimos días:

1. "Hace mil millones de años la simbiosis entre dos células creó un reino nuevo en la naturaleza. Estamos intentando replicar el encuentro"

Este artículo se refiere a cómo las simbioses entre diferentes especies pueden dar lugar a nuevos seres y ecosistemas. El autor analiza cómo esta tendencia ha sido observable en la naturaleza durante millones de años y si es posible que los humanos puedan aprender de estos ejemplos.

2. "La energía eólica se las prometía felices en su expansión por Colombia. Hasta que se topó con las comunidades indígenas"

Este artículo explora cómo la expansión de la energía eólica en Colombia ha sido contrariada por la oposición de algunas comunidades indígenas que han expresado preocupación sobre los impactos ambientales y sociales de estas plantas. El autor analiza las complejidades involucradas en encontrar un equilibrio entre el desarrollo energético sostenible y la protección del medio ambiente y los derechos humanos.

3. "Se suponía que este modelo que OpenAI presenta como mejora frente a GPT-4o no llegaría a ChatGPT. Pero ahora está aquí"

Este artículo discute sobre las últimas noticias en cuanto al desarrollo de modelos de inteligencia artificial avanzada. El autor analiza cómo algunos expertos creen que el nuevo modelo puede mejorar significativamente las capacidades del modelo anterior, lo cual podría tener implicaciones importantes para la industria del AI.

4. "Tesla acaba de mostrar lo que Optimus puede hacer sin estar atado: su forma de moverse ya empieza a resultar familiar"

Este artículo presenta las últimas noticias en cuanto al desarrollo del modelo de autónomo de Tesla, llamado Optimus. El autor analiza cómo el modelo parece demostrar cierta eficiencia y estabilidad en sus pruebas, lo que podría significar avances significativos para la industria automotriz.

5. "MediaMarkt lanza un nuevo Apple Days con precios mínimos históricos de la tienda. iPhone, MacBook, iPad y mucho más en oferta"

Este artículo anuncia la llegada de una nueva promoción de MediaMarkt, que ofrece precios reducidos a varios productos de Apple, incluyendo el iPhone, el MacBook y el iPad. El autor analiza las oportunidades y ventajas que este tipo de promociones pueden ofrecer a los consumidores.

6. "Anker Eufy Omni E28, análisis: érase una vez un aspirador 'transformer' tan imperfecto como recomendable"

Este artículo ofrece un análisis detallado del Anker Eufy Omni E28, un aspirador que se caracteriza por su capacidad para limpiar sin necesidad de agua. El autor analiza las características y los pros y contras de este producto, lo cual puede ser útil para aquellos que buscan una opción más eficiente y sostenible para la limpieza.

7. "Arabia Saudí quiere convertirse en una nueva potencia en centros de datos. No está nada claro que pueda hacerlo"

Este artículo analiza cómo Arabia Saudí pretende establecerse como un importante centro de datos en el mundo, lo cual podría tener implicaciones significativas para la economía del país y su influencia global.

8. "Ucrania ha convertido drones en soldados armados contra Rusia. Ya no hace falta 'sacrificarlos' porque llevan lanzagranadas"

Este artículo presentaba las últimas noticias sobre el uso de drones por parte de Ucrania durante la guerra con Rusia. El autor analiza cómo estos drones están siendo utilizados para combatir a los militares rusos y cómo esto podría ser un paso hacia una nueva forma de guerra.

9. "Tesla se desvive para 'liberar' a su CEO, Elon Musk, del control totalitario del consejo directivo"

Este artículo discute sobre la situación de Tesla, donde Elon Musk tiene un control significativo sobre la empresa gracias a su posición como CEO y miembro del consejo directivo. El autor analiza las críticas que se han lanzado contra este control y cómo esto podría afectar al futuro de la empresa.

10. "La seguridad online es cada vez más importante, por lo que debes estar preparada para proteger tu privacidad"

Este artículo destaca la importancia de la seguridad en línea y cómo es fundamental tomar medidas para protegerse contra las amenazas cibernéticas. El autor ofrece consejos y recomendaciones para aquellos que desean mantener su privacidad y seguridad en línea.

Espero que estos artículos te sean útiles.

In [18]:
display_summary("https://cnn.com")

**Website Summary: CNN Breaking News**
=====================================

*   **Primary Content:** Breaking news, latest news, and videos on various topics such as politics, business, health, entertainment, style, travel, sports, science, climate, and more.
*   **News Highlights:**
    *   Putin's no-show reveals who is calling the shots in Ukraine-Russia talks
    *   Trump says US is 'very close' to a nuclear deal after Iran 'agreed' to its terms
    *   TikTok beauty influencer shot dead during live stream in Mexico
    *   Argentina orders immigration crackdown with new decree to 'make Argentina great again'
    *   NASA revives 'dead' thrusters on the farthest spacecraft from Earth
*   **Special Sections:** Space and Science, Global Travel, Global Business, Style, Sports, and more.
*   **Live Updates:** Live updates on various global events, including Ukraine-Russia talks, US Supreme Court arguments, and more.

In [13]:
display_summary("https://anthropic.com")

**Summary of Anthropic Website**

### Mission and Values

* Build AI to serve humanity's long-term well-being.
* Focus on responsible AI development with human benefit at its foundation.

### Research and Publications

* Research articles and papers on AI safety, alignment science, and societal impacts.
* Featured publications include:
	+ "Tracing the thoughts of a large language model" (Interpretability)
	+ "Anthropic Economic Index"
	+ "Claude 3.7 Sonnet and Claude Code"

### Products and Services

* Claude: an AI-powered platform for building custom experiences.
* Download apps: available on various platforms, including Apple App Store and Google Play.

### Initiatives and Commitments

* Responsible scaling policy: ensures responsible growth of AI systems.
* Transparency: committed to open communication about research and developments.

### Careers and Education

* Anthropic Academy: learn to build with Claude.
* Open roles: see available positions on the website.

### News and Announcements

* ISO 42001 certification: Anthropic has achieved this certification for its products.
* Recent updates:
	+ Claude 3.7 Sonnet is now available.
	+ Alignment faking in large language models (Product).
	+ Introducing the Model Context Protocol (Product).

In [19]:
display_summary("https://www.eximmachinery.com")

### Exim Machinery Website Summary

#### Overview
Exim Machinery is a website that appears to be a used vehicle dealership, offering a variety of vehicles for sale. The website provides features such as search functionality, filtering options (e.g., price range, mileage), and detailed information about each vehicle.

#### Featured Section
The "Vehículos" section showcases multiple vehicles with their respective details, including:

* Vehicle reference number
* Price
* Title
* Matriculation date and construction date
* Kilometrage and hours of use
* Eje configuration and transmission type

#### News/Announcements
A section titled "Novedades" indicates the presence of recent announcements or news. However, no specific content is provided in this summary.

#### Company Information
The website includes information about the company's values, such as:

* Asistencia de calidad (quality assistance)
* Photo empresa (company photo)
* Nuestros colaboradores (our colleagues)

#### Contact Information
Contact details are listed at the bottom of the page, including:

* Phone number: 0034 961 539 366
* Email address: exim@eximmachinery.com
* Address: C/ Pedrapiquers s/n - Pol Ribarroja Sector 13 - 46190 Ribarroja de Turia - Valencia - ESPAÑA.

#### Registration Form
A registration form is present at the top of the page, allowing users to create an account or log in.